# 04 · Interactive demo: "what could this place look like after a disaster?"

1. Search for a place or pan the map, then draw a **rectangle** (≤ 25 km², inside Kerala).
2. Choose the hazard and a rainfall scenario, press **Generate**.
3. Press ▶ to crossfade from today's image (severity 0) to increasingly severe outcomes.

Generated images are plausible scenarios for hazard communication, **not predictions**.
Severities above the range seen in training are marked as extrapolation.
Runtime: GPU recommended (CPU works, slower). Set `GEE_PROJECT` first.

In [ ]:
from google.colab import drive, output

GEE_PROJECT = "your-gee-project-id"  # <- change this
REPO_URL = "https://github.com/muhammedshahidsds25-collab/disaster-gen.git"

output.enable_custom_widget_manager()  # needed for the map widget in Colab
drive.mount("/content/drive")
!git clone -q {REPO_URL} /content/disaster-gen || git -C /content/disaster-gen pull -q
%cd /content/disaster-gen
!pip install -q -e ".[gee]"

In [ ]:
import io
from pathlib import Path

import ee
import geemap
import ipywidgets as widgets
import numpy as np
import torch
from PIL import Image

from disastergen.config import load_config
from disastergen.data.dataset import ChannelStats
from disastergen.demo import (
    crossfade,
    fetch_area,
    generate_sweep,
    is_extrapolation,
    rain_plane_cond,
)
from disastergen.evaluate import plot_severity_sweep, rgb_uint8
from disastergen.models import UNetGenerator

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
CKPT_DIR = DRIVE_ROOT / "checkpoints"
FIG_DIR = DRIVE_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

stats = ChannelStats.load(CKPT_DIR / "stats.json")
g = UNetGenerator(cfg.model.in_channels, cfg.model.out_channels, cfg.model.ngf, cfg.model.num_downs)
g.load_state_dict(torch.load(CKPT_DIR / "best_g.pt", map_location=device, weights_only=True))
g = g.to(device).eval()
TRAIN_RAIN_MEAN = float(stats.mean[5])  # channel order: 4 bands, slope, rain, soil
CROSSFADE_STEPS = 30

In [ ]:
def run_demo(bbox, hazard: int, rain_mm: float):
    """Return (frames, severities): today's image, then crossfaded generated outcomes."""
    before, slope, soil = fetch_area(bbox, cfg)
    cond = rain_plane_cond(slope, soil, rain_mm)
    key_frames = generate_sweep(g, before, cond, hazard, cfg.demo.sweep, stats,
                                cfg.eval.rgb_max, cfg.data.tile_size, device)
    today = rgb_uint8(torch.from_numpy(before), cfg.eval.rgb_max).permute(1, 2, 0).numpy()
    key_severities = [0.0, *cfg.demo.sweep]
    frames = crossfade([today, *key_frames], CROSSFADE_STEPS)
    positions = np.linspace(0, len(key_severities) - 1, CROSSFADE_STEPS)
    severities = np.interp(positions, range(len(key_severities)), key_severities)
    return frames, severities, today, key_frames


def to_png(frame: np.ndarray) -> bytes:
    buffer = io.BytesIO()
    Image.fromarray(frame).save(buffer, format="PNG")
    return buffer.getvalue()


m = geemap.Map(center=(10.5, 76.5), zoom=8)
hazard = widgets.ToggleButtons(options=[("Landslide", 1), ("Flood", 0)], value=1)
rain = widgets.FloatSlider(value=TRAIN_RAIN_MEAN, min=0, max=800, step=10,
                           description="Rain (mm/7d)", continuous_update=False)
generate = widgets.Button(description="Generate", button_style="primary")
status = widgets.HTML()
picture = widgets.Image(format="png", width=400)
play = widgets.Play(min=0, max=CROSSFADE_STEPS - 1, interval=120)
scrub = widgets.IntSlider(min=0, max=CROSSFADE_STEPS - 1, description="Frame")
widgets.jslink((play, "value"), (scrub, "value"))
state = {"frames": [], "severities": []}


def show(_=None):
    if not state["frames"]:
        return
    i = scrub.value
    picture.value = to_png(state["frames"][i])
    severity = state["severities"][i]
    warning = " ⚠️ <b>extrapolation</b>" if is_extrapolation(severity, cfg.demo) else ""
    status.value = f"Severity {severity:.2f}{warning}"


def on_generate(_):
    if m.user_roi is None:
        status.value = "Draw a rectangle on the map first."
        return
    lons, lats = zip(*m.user_roi.bounds().coordinates().getInfo()[0], strict=True)
    bbox = (min(lons), min(lats), max(lons), max(lats))
    status.value = "Fetching satellite data and generating…"
    try:
        state["frames"], state["severities"], *_ = run_demo(bbox, hazard.value, rain.value)
    except ValueError as error:
        status.value = f"⚠️ {error}"
        return
    scrub.value = 0
    show()


generate.on_click(on_generate)
scrub.observe(show, names="value")
widgets.VBox([m, widgets.HBox([hazard, rain, generate]), widgets.HBox([play, scrub]),
              status, picture])

## Saved examples (static, so they show in the committed notebook)

In [ ]:
EXAMPLES = {
    "wayanad": (76.13, 11.46, 76.15, 11.48),
    "pettimudi": (77.00, 10.16, 77.02, 10.18),
}
for name, bbox in EXAMPLES.items():
    _, _, today, key_frames = run_demo(bbox, hazard=1, rain_mm=TRAIN_RAIN_MEAN)
    as_tensor = [torch.from_numpy(f).permute(2, 0, 1) for f in key_frames]
    display(plot_severity_sweep(torch.from_numpy(today).permute(2, 0, 1), as_tensor,
                                cfg.demo.sweep, FIG_DIR / f"demo_{name}.png"))